In [2]:
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

# For embeddings
from sentence_transformers import SentenceTransformer

# For vector database
import chromadb

# For progress tracking
from tqdm import tqdm

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [4]:
def simple_chunk_text(text, chunk_size=500, overlap=50):
    """
    Simple text chunking without LangChain
    
    Args:
        text (str): Text to chunk
        chunk_size (int): Size of each chunk (characters)
        overlap (int): Overlap between chunks (characters)
    
    Returns:
        list: List of text chunks
    """
    if not isinstance(text, str) or len(text) == 0:
        return []
    
    # If text is shorter than chunk_size, return it as one chunk
    if len(text) <= chunk_size:
        return [text.strip()]
    
    chunks = []
    start = 0
    
    while start < len(text):
        # Find the end position
        end = start + chunk_size
        
        # If we're at the end, just take the rest
        if end >= len(text):
            chunks.append(text[start:].strip())
            break
        
        # Try to break at a sentence boundary (., !, ?)
        # Look for the last period, exclamation, or question mark within the chunk
        search_text = text[start:end]
        last_period = search_text.rfind('.')
        last_exclamation = search_text.rfind('!')
        last_question = search_text.rfind('?')
        last_newline = search_text.rfind('\n')
        
        # Find the best break point
        break_positions = [last_period, last_exclamation, last_question, last_newline]
        # Add a space as fallback
        break_positions.append(search_text.rfind(' '))
        
        # Find the best position (closest to chunk_size but not too far)
        best_pos = max([p for p in break_positions if p > chunk_size * 0.3])
        
        if best_pos > 0:
            end = start + best_pos + 1
        else:
            # If no good break point, just cut at chunk_size
            end = start + chunk_size
        
        # Extract the chunk
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        
        # Move start position (with overlap)
        start = end - overlap
    
    return chunks

# Test the chunking function
test_text = """This is a test sentence. Here is another one. And a third one for good measure. 
This is a longer paragraph to test the chunking. It should break at sentence boundaries.
Let's see if it works properly with multiple sentences and periods."""

chunks = simple_chunk_text(test_text, chunk_size=100, overlap=20)
print(f"Test: {len(chunks)} chunks created")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk[:50]}...")

Test: 3 chunks created
Chunk 1: This is a test sentence. Here is another one. And ...
Chunk 2: . 
This is a longer paragraph to test the chunking...
Chunk 3: e boundaries.
Let's see if it works properly with ...


In [5]:
# Load the cleaned dataset from Task 1
df = pd.read_csv('../data/processed/filtered_complaints.csv')
print(f"Loaded dataset: {len(df):,} complaints")

print("\nProduct distribution:")
print(df['Product'].value_counts())

# Preview
df.head(2)

Loaded dataset: 454,472 complaints

Product distribution:
Product
Credit card        189334
Savings account    140319
Money transfer      98685
Personal loan       26134
Name: count, dtype: int64


,Product,Issue,Sub-issue,Company,State,Date received,narrative
0,Credit card,Getting a credit card,Card opened without my consent or knowledge,"CITIBANK, N.A.",TX,2025-06-13,a xxxx xxxx card was opened under my name by a...
1,Savings account,Managing an account,Deposits and withdrawals,WELLS FARGO & COMPANY,ID,2025-06-13,i made the mistake of using my wellsfargo debi...


In [6]:
print("=" * 50)
print("CREATING STRATIFIED SAMPLE")
print("=" * 50)

# Set sample size
SAMPLE_SIZE = 10000  # You can increase to 15,000 if you want

# Function to create stratified sample
def create_stratified_sample(df, sample_size, product_col='Product'):
    """
    Create a stratified sample maintaining product distribution
    """
    # Calculate number of samples per product
    product_counts = df[product_col].value_counts()
    proportions = product_counts / len(df)
    
    # Calculate sample sizes per product
    sample_sizes = (proportions * sample_size).round().astype(int)
    
    # Ensure we don't sample more than available
    sample_sizes = sample_sizes.clip(upper=product_counts)
    
    # Adjust if total is less than sample_size
    total_samples = sample_sizes.sum()
    if total_samples < sample_size:
        # Add remaining to the largest product
        largest_product = sample_sizes.idxmax()
        sample_sizes[largest_product] += (sample_size - total_samples)
    
    print("Sample sizes per product:")
    print(sample_sizes)
    
    # Sample from each product
    sampled_dfs = []
    for product, n_samples in sample_sizes.items():
        product_df = df[df[product_col] == product]
        if n_samples > 0:
            sampled = product_df.sample(n=n_samples, random_state=42)
            sampled_dfs.append(sampled)
    
    # Combine all samples
    sampled_df = pd.concat(sampled_dfs, ignore_index=True)
    
    print(f"\nCreated stratified sample: {len(sampled_df):,} complaints")
    print("\nSample distribution:")
    print(sampled_df['Product'].value_counts())
    
    return sampled_df

# Create the sample
df_sample = create_stratified_sample(df, SAMPLE_SIZE)

# Save sample for reference
os.makedirs('../data/processed', exist_ok=True)
df_sample.to_csv('../data/processed/sample_complaints.csv', index=False)
print("\nSample saved to: data/processed/sample_complaints.csv")

CREATING STRATIFIED SAMPLE
Sample sizes per product:
Product
Credit card        4166
Savings account    3088
Money transfer     2171
Personal loan       575
Name: count, dtype: int64

Created stratified sample: 10,000 complaints

Sample distribution:
Product
Credit card        4166
Savings account    3088
Money transfer     2171
Personal loan       575
Name: count, dtype: int64

Sample saved to: data/processed/sample_complaints.csv


In [7]:
print("=" * 50)
print("PROCESSING ALL NARRATIVES INTO CHUNKS")
print("=" * 50)

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# Create lists to store chunks and metadata
chunks = []
metadata = []

# Iterate through each row
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Processing complaints"):
    narrative = row['narrative']
    
    # Skip if narrative is empty
    if not isinstance(narrative, str) or len(narrative.strip()) == 0:
        continue
    
    # Split into chunks
    narrative_chunks = simple_chunk_text(narrative, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    
    # Store each chunk with metadata
    for chunk_idx, chunk in enumerate(narrative_chunks):
        chunks.append(chunk)
        metadata.append({
            'complaint_id': idx,
            'product': row['Product'],
            'issue': row.get('Issue', 'Unknown'),
            'state': row.get('State', 'Unknown'),
            'chunk_index': chunk_idx,
            'total_chunks': len(narrative_chunks)
        })

print(f"\nCreated {len(chunks):,} chunks from {len(df_sample)} complaints")
print(f"Average chunks per complaint: {len(chunks)/len(df_sample):.2f}")

# Create DataFrame with chunks
chunk_df = pd.DataFrame({
    'chunk': chunks,
    'metadata': metadata
})

chunk_df.head(3)

PROCESSING ALL NARRATIVES INTO CHUNKS


Processing complaints: 100%|██████████| 10000/10000 [00:01<00:00, 7764.99it/s]



Created 28,866 chunks from 10000 complaints
Average chunks per complaint: 2.89


,chunk,metadata
0,my soon to be ex wife took out several credit ...,"{'complaint_id': 0, 'product': 'Credit card', ..."
1,after informing bank of america of my identity...,"{'complaint_id': 1, 'product': 'Credit card', ..."
2,i am deeply troubled by the inclusion of this ...,"{'complaint_id': 2, 'product': 'Credit card', ..."


In [8]:
print("=" * 50)
print("GENERATING EMBEDDINGS")
print("=" * 50)

# Load the embedding model
print("Loading embedding model: all-MiniLM-L6-v2")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded! Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Generate embeddings in batches
batch_size = 32
embeddings = []

print(f"\nGenerating embeddings for {len(chunks):,} chunks...")

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding chunks"):
    batch_chunks = chunks[i:i+batch_size]
    batch_embeddings = model.encode(
        batch_chunks,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    embeddings.extend(batch_embeddings)

# Convert to numpy array
embeddings_array = np.array(embeddings)
print(f"\nEmbeddings generated!")
print(f"  - Shape: {embeddings_array.shape}")
print(f"  - Dimension: {embeddings_array.shape[1]}")

GENERATING EMBEDDINGS
Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded! Embedding dimension: 384

Generating embeddings for 28,866 chunks...


Embedding chunks: 100%|██████████| 903/903 [06:42<00:00,  2.24it/s]


Embeddings generated!
  - Shape: (28866, 384)
  - Dimension: 384


In [10]:
print("=" * 50)
print("SAVING EMBEDDINGS AND METADATA")
print("=" * 50)

# Create directory for vector store
os.makedirs('../vector_store', exist_ok=True)

# Save embeddings as numpy array
np.save('../vector_store/embeddings.npy', embeddings_array)
print(f"Saved embeddings: vector_store/embeddings.npy")

# Save metadata as DataFrame
chunk_df.to_parquet('../vector_store/chunks_metadata.parquet', index=False)
print(f"Saved metadata: vector_store/chunks_metadata.parquet")

# Also save as CSV for easy viewing
chunk_df.to_csv('../vector_store/chunks_metadata.csv', index=False)
print(f"Saved CSV: vector_store/chunks_metadata.csv")

# Save sample of chunks for inspection
print(f"\nSample of saved data:")
print(chunk_df[['chunk', 'metadata']].head(3))

SAVING EMBEDDINGS AND METADATA
Saved embeddings: vector_store/embeddings.npy
Saved metadata: vector_store/chunks_metadata.parquet
Saved CSV: vector_store/chunks_metadata.csv

Sample of saved data:
                                               chunk  \
0  my soon to be ex wife took out several credit ...   
1  after informing bank of america of my identity...   
2  i am deeply troubled by the inclusion of this ...   

                                            metadata  
0  {'complaint_id': 0, 'product': 'Credit card', ...  
1  {'complaint_id': 1, 'product': 'Credit card', ...  
2  {'complaint_id': 2, 'product': 'Credit card', ...  


In [11]:
print("=" * 50)
print("CREATING CHROMADB VECTOR STORE")
print("=" * 50)

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="../vector_store/chroma_db")

# Create or get collection
collection_name = "complaints"
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

print(f"Collection created: {collection_name}")
print(f"Collection count: {collection.count()}")

# Add chunks to ChromaDB in batches
batch_size = 100
print(f"\nAdding {len(chunks):,} chunks to ChromaDB...")

for i in tqdm(range(0, len(chunks), batch_size), desc="Adding to ChromaDB"):
    batch_end = min(i + batch_size, len(chunks))
    
    # Prepare batch data
    batch_ids = [f"chunk_{i + j}" for j in range(batch_end - i)]
    batch_chunks = chunks[i:batch_end]
    batch_embeddings = embeddings_array[i:batch_end].tolist()
    batch_metadata = metadata[i:batch_end]
    
    # Add to collection
    collection.add(
        documents=batch_chunks,
        embeddings=batch_embeddings,
        metadatas=batch_metadata,
        ids=batch_ids
    )

print(f"\nAll chunks added to ChromaDB!")
print(f"Total items in collection: {collection.count()}")

CREATING CHROMADB VECTOR STORE
Collection created: complaints
Collection count: 0

Adding 28,866 chunks to ChromaDB...


Adding to ChromaDB: 100%|██████████| 289/289 [00:35<00:00,  8.23it/s]


All chunks added to ChromaDB!
Total items in collection: 28866


In [12]:
print("=" * 50)
print("TESTING VECTOR STORE")
print("=" * 50)

# Test query
test_query = "Why are customers unhappy with credit cards?"

# Generate query embedding
query_embedding = model.encode(test_query).tolist()

# Search
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Query: {test_query}")
print("\nTop 3 results:")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    print(f"\nResult {i+1} (Distance: {dist:.4f}):")
    print(f"  Product: {meta.get('product', 'Unknown')}")
    print(f"  Preview: {doc[:200]}...")

TESTING VECTOR STORE
Query: Why are customers unhappy with credit cards?

Top 3 results:

Result 1 (Distance: 0.3519):
  Product: Credit card
  Preview: first time it s like a scheme . as a good credit card company as i ve went through these so many years they ve done a lot and trying to do more and more for their consumers to win business and win rep...

Result 2 (Distance: 0.3642):
  Product: Credit card
  Preview: t either the credit card company is not reporting correctly or the reporting companies are not reporting correctly. either way ... me the consumer is negatively and unfairly effected. the purpose of t...

Result 3 (Distance: 0.3719):
  Product: Credit card
  Preview: ists. the credit card companies and the banks are stealing money from people and then telling them after the fact and i m sure in some instances people do not expect this and use the cards and then ge...


In [13]:
print("=" * 50)
print("TASK 2 COMPLETE - SUMMARY")
print("=" * 50)

print(f"Sample size: {len(df_sample):,} complaints")
print(f"Total chunks: {len(chunks):,}")
print(f"Embedding dimension: {embeddings_array.shape[1]}")
print(f"ChromaDB collection: {collection_name}")
print(f"Total vectors in DB: {collection.count()}")

print("\nFiles saved:")
print("  - vector_store/embeddings.npy")
print("  - vector_store/chunks_metadata.parquet")
print("  - vector_store/chunks_metadata.csv")
print("  - vector_store/chroma_db/ (ChromaDB persistent storage)")
print("  - data/processed/sample_complaints.csv")

print("\nTask 2 completed successfully!")

TASK 2 COMPLETE - SUMMARY
Sample size: 10,000 complaints
Total chunks: 28,866
Embedding dimension: 384
ChromaDB collection: complaints
Total vectors in DB: 28866

Files saved:
  - vector_store/embeddings.npy
  - vector_store/chunks_metadata.parquet
  - vector_store/chunks_metadata.csv
  - vector_store/chroma_db/ (ChromaDB persistent storage)
  - data/processed/sample_complaints.csv

Task 2 completed successfully!
